---
# DNN : `ozone_complet` (avec `Keras` et `Optuna`)
---

## Packages

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import keras
from keras import callbacks, layers, models

import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances

---
## 1. Données
---

### 1.1. Importation

In [2]:
# En local :
directory = '/Users/vincentlefieux/Dropbox/Docs_ACADEMIQUE/Data/'

# Sur Google collab ou Onyxia (sur un répertoire temporaire) :
# directory = ''

# Sur Google collab (sur le drive) :
# from google.colab import drive
# drive.mount('/content/drive')
# directory = '/content/drive/MyDrive/Data/'

In [3]:
data = pd.read_csv(directory + 'ozone_complet.csv',
                   header    = 0,
                   index_col = 0,
                   sep       = ';',
                   decimal   = ',')

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1464 entries, 19950401 to 20020930
Data columns (total 23 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   maxO3   1391 non-null   float64
 1   T6      1463 non-null   float64
 2   T9      1463 non-null   float64
 3   T12     1463 non-null   float64
 4   T15     1463 non-null   float64
 5   T18     1463 non-null   float64
 6   Ne6     1462 non-null   float64
 7   Ne9     1462 non-null   float64
 8   Ne12    1462 non-null   float64
 9   Ne15    1462 non-null   float64
 10  Ne18    1459 non-null   float64
 11  Vdir6   1463 non-null   float64
 12  Vvit6   1463 non-null   float64
 13  Vdir9   1463 non-null   float64
 14  Vvit9   1463 non-null   float64
 15  Vdir12  1463 non-null   float64
 16  Vvit12  1463 non-null   float64
 17  Vdir15  1463 non-null   float64
 18  Vvit15  1463 non-null   float64
 19  Vdir18  1463 non-null   float64
 20  Vvit18  1463 non-null   float64
 21  Vx      1463 non-null   float64

In [5]:
data.head()

,maxO3,T6,T9,T12,T15,T18,Ne6,Ne9,Ne12,Ne15,...,Vdir9,Vvit9,Vdir12,Vvit12,Vdir15,Vvit15,Vdir18,Vvit18,Vx,maxO3v
date,,,,,,,,,,,,,,,,,,,,,
19950401,47.6,10.1,11.6,13.3,13.6,12.2,8.0,8.0,8.0,8.0,...,290.0,4.0,300.0,4.0,340.0,4.0,20.0,4.0,-3.4641,62.2
19950402,56.2,9.5,9.4,13.8,17.4,16.3,8.0,8.0,7.0,0.0,...,160.0,2.0,180.0,3.0,110.0,1.0,350.0,2.0,0.0000,47.6
19950403,61.8,3.6,8.0,16.8,21.5,20.2,4.0,5.0,2.0,2.0,...,20.0,2.0,340.0,1.0,170.0,2.0,170.0,3.0,-0.3420,56.2
19950404,50.8,9.5,10.5,11.4,12.2,11.4,8.0,7.0,7.0,7.0,...,10.0,4.0,350.0,3.0,350.0,3.0,350.0,4.0,-0.5209,61.8
19950405,59.8,9.8,10.8,13.8,14.3,13.3,8.0,7.0,8.0,8.0,...,340.0,2.0,280.0,1.0,320.0,3.0,350.0,4.0,-0.9848,50.8


### 1.2. Gestion des données manquantes

In [6]:
missing_percentage = data.isna().mean() * 100

print('MISSING VALUES :')
if missing_percentage[missing_percentage != 0].empty:
    print('No')
else:
    print(missing_percentage[missing_percentage != 0].sort_values(ascending=False))

MISSING VALUES :
maxO3     4.986339
maxO3v    4.986339
Ne18      0.341530
Ne6       0.136612
Ne9       0.136612
Ne12      0.136612
Ne15      0.136612
T9        0.068306
Vdir12    0.068306
Vx        0.068306
Vvit18    0.068306
Vdir18    0.068306
Vvit15    0.068306
Vdir15    0.068306
Vvit12    0.068306
Vvit6     0.068306
Vvit9     0.068306
Vdir9     0.068306
T12       0.068306
T6        0.068306
T18       0.068306
T15       0.068306
Vdir6     0.068306
dtype: float64


In [7]:
data.dropna(inplace=True)

### 1.3. Gestion des variables

In [8]:
target = 'maxO3'

y = data[target]
X = data.drop(target, axis=1)

### 1.4. Création des échantillons de validation et test

In [9]:
test_portion  = 1/5
valid_portion = 1/5

X_train_valid, X_test, y_train_valid, y_test = train_test_split(X, y, test_size=test_portion)

X_train, X_valid, y_train, y_valid = train_test_split(X_train_valid, y_train_valid, test_size=valid_portion)

print('Dimensions de X_train :', X_train.shape)
print('Dimensions de X_valid :', X_valid.shape)
print('Dimensions de X_test  :', X_test.shape)

print('Dimensions de y_train :', y_train.shape)
print('Dimensions de y_valid :', y_valid.shape)
print('Dimensions de y_test  :', y_test.shape)

Dimensions de X_train : (873, 22)
Dimensions de X_valid : (219, 22)
Dimensions de X_test  : (274, 22)
Dimensions de y_train : (873,)
Dimensions de y_valid : (219,)
Dimensions de y_test  : (274,)


### 1.5. Normalisation des covariables

On normalise (centrage-réduction) les covariables :

In [10]:
scaler = StandardScaler(with_mean=True, with_std=True)
scaler.fit(X_train)

X_train_norm = scaler.transform(X_train)
X_valid_norm = scaler.transform(X_valid)
X_test_norm  = scaler.transform(X_test)

---
## 2. DNN
---

### 2.1. Architecture et optimiseur

In [11]:
dim_inputs  = (X_train_norm.shape[1],)
dim_outputs = 1

n_layers_min = 1
n_layers_max = 3

n_units_min  = 32
n_units_max  = 128
n_units_step = 32

dropout_min  = 0.2
dropout_max  = 0.2
dropout_step = 0.1

activation_hidden = 'relu'
activation_output = 'linear'

optimizer = 'adam'
loss      = 'mse'
metrics   = ['mae']

monitor  = 'val_loss'
mode     = 'min'
patience = 10

batch_size   = 300
n_epochs_max = 200
verbose      = 0

def objective(trial):
    num_layers = trial.suggest_int('num_layers', n_layers_min, n_layers_max)
    layers_units = []
    for i in range(num_layers):
        units = trial.suggest_int(f'units_layer_{i}', n_units_min, n_units_max, step=n_units_step)
        layers_units.append(units)
    dropout_rate = trial.suggest_float('dropout_rate', dropout_min, dropout_max, step=dropout_step)
    # learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    
    model = models.Sequential()
    for units in layers_units:
        model.add(layers.Dense(units, activation=activation_hidden))
        model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(dim_outputs, activation=activation_output))
    
    model.compile(optimizer=optimizer,
                  # optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                  loss=loss,
                  metrics=metrics)
    
    callback = keras.callbacks.EarlyStopping(monitor              = monitor,
                                             mode                 = mode,
                                             patience             = patience,
                                             restore_best_weights = True)
    
    hist = model.fit(X_train_norm,
                     y_train,
                     validation_data = (X_valid_norm, y_valid),
                     batch_size      = batch_size,
                     epochs          = n_epochs_max,
                     verbose         = verbose,
                     callbacks       = [callback])
    
    score_valid = model.evaluate(X_valid_norm, y_valid, verbose=0)
    loss_valid = score_valid[0]
    
    return loss_valid

### 2.2. Entraînement

In [12]:
n_trials = 50

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=n_trials)

print('Meilleurs hyperparamètres :', study.best_params)
print('Meilleur MSE (validation) :', study.best_value)

plot_optimization_history(study).show()
plot_param_importances(study).show()

# Pour la sauvegarde et le chargement de l'étude
# import joblib
# joblib.dump(study, 'optuna_study.pkl') # Sauvegarde
# study = joblib.load('optuna_study.pkl') # Chargement

[I 2025-12-31 16:54:50,842] A new study created in memory with name: no-name-2ffda4aa-31f4-4779-9092-e9905e647887
[I 2025-12-31 16:54:54,624] Trial 0 finished with value: 577.23046875 and parameters: {'num_layers': 1, 'units_layer_0': 128, 'dropout_rate': 0.2}. Best is trial 0 with value: 577.23046875.
[I 2025-12-31 16:54:58,521] Trial 1 finished with value: 558.8435668945312 and parameters: {'num_layers': 1, 'units_layer_0': 128, 'dropout_rate': 0.2}. Best is trial 1 with value: 558.8435668945312.
[I 2025-12-31 16:55:02,012] Trial 2 finished with value: 196.91510009765625 and parameters: {'num_layers': 3, 'units_layer_0': 128, 'units_layer_1': 96, 'units_layer_2': 32, 'dropout_rate': 0.2}. Best is trial 2 with value: 196.91510009765625.
[I 2025-12-31 16:55:06,063] Trial 3 finished with value: 211.2817840576172 and parameters: {'num_layers': 2, 'units_layer_0': 96, 'units_layer_1': 96, 'dropout_rate': 0.2}. Best is trial 2 with value: 196.91510009765625.
[I 2025-12-31 16:55:09,891] Tri

Meilleurs hyperparamètres : {'num_layers': 3, 'units_layer_0': 96, 'units_layer_1': 96, 'units_layer_2': 96, 'dropout_rate': 0.2}
Meilleur MSE (validation) : 171.53297424316406


### 2.3. Prévisions

In [13]:
best_params = study.best_trial.params

model = models.Sequential()
model.add(layers.Input(dim_inputs))
for i in range(best_params['num_layers']):
    units = best_params[f'units_layer_{i}']
    model.add(layers.Dense(units, activation=activation_hidden))
    model.add(layers.Dropout(best_params['dropout_rate']))
model.add(layers.Dense(dim_outputs, activation=activation_output))

model.compile(optimizer=optimizer,
              # optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['learning_rate']),
              loss=loss,
              metrics=metrics)

callback = callbacks.EarlyStopping(monitor              = monitor,
                                   mode                 = mode,
                                   patience             = patience,
                                   restore_best_weights = True)

hist = model.fit(X_train_norm,
                 y_train,
                 batch_size      = batch_size,
                 validation_data = (X_valid_norm, y_valid),
                 epochs          = n_epochs_max,
                 callbacks       = [callback])

Epoch 1/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 7564.9619 - mae: 84.0239 - val_loss: 7636.7729 - val_mae: 84.2155
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 7652.8228 - mae: 84.3450 - val_loss: 7526.7358 - val_mae: 83.5681
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 7419.4189 - mae: 83.0828 - val_loss: 7400.2070 - val_mae: 82.8169
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 7443.9253 - mae: 83.0136 - val_loss: 7243.7896 - val_mae: 81.8820
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 7049.0688 - mae: 80.8731 - val_loss: 7047.4761 - val_mae: 80.6962
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 6892.5229 - mae: 79.8177 - val_loss: 6797.5005 - val_mae: 79.1642
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 6580.7520 - mae: 78.0666 - val_loss: 6478.7168 - val_mae: 77.1683
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 6421.3330 - mae: 76.7794 - val_loss: 6075.7520 - val_mae: 74.5702
Epoch 9/200
3/3

In [14]:
y_test_pred = model.predict(X_test_norm)
y_test_pred[0:5]

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step  


array([[ 76.64089 ],
       [ 82.81827 ],
       [112.75243 ],
       [ 65.26056 ],
       [ 86.677795]], dtype=float32)

In [15]:
score_test = model.evaluate(X_test_norm, y_test, verbose=0)
print(f'MSE test : {score_test[0]:4.4f}')
print(f'MAE test : {score_test[1]:4.4f}')

MSE test : 224.2109
MAE test : 11.5405
